In [1]:
import cellphonedb
from cellphonedb.src.core.methods import cpdb_analysis_method
import pandas as pd
import glob
import os
from IPython.display import HTML, display
from cellphonedb.utils import db_releases_utils


2

In [2]:
# -- Version of the databse
cpdb_version = 'v4.1.0'

In [3]:
from cellphonedb.utils import db_utils

All_Samples_aLL_markers_06_07_2023.tsv
All_samples_UA_microenv_June7th.tsv
All_samples_Unvaxed_filtered_meta_June7th.tsv
Annotatedd_ALL_markers_Jun6th2023.csv


In [12]:
####Cd8 SUCCESSFULLY GET THE CELLPHONEDB
cpdb_file_path = 'v4.1.0/cellphonedb.zip'
meta_file_path = 'All_samples/Unvax_A/All_samples_Unvaxed_filtered_meta_June7th.tsv'
counts_file_path = 'All_samples/all_Samples_seurat_counts.h5ad'
microenvs_file_path = 'All_samples/Unvax_A/All_samples_UA_microenv_June7th.tsv'
degs_file_path = 'All_samples/Unvax_A/All_Samples_aLL_markers_06_07_2023.tsv'
out_path = 'All_sample_results/Unvaxed/method2'

In [13]:
metadata = pd.read_csv(meta_file_path, sep = '\t')
metadata.head(3)

#(metadata)['major_cell_type']
#metadata["major_cell_type"][1:3]

,barcode_sample,major_cell_type,state
0,1_MEM1_12_4_20_AAACCTGCACATGGGA-1,Unvax_A_CD8,Unvax_A
1,1_MEM1_12_4_20_AAACCTGCAGGATTGG-1,Unvax_A_Macrophage,Unvax_A
2,1_MEM1_12_4_20_AAACCTGGTAAACCTC-1,Unvax_A_CD4,Unvax_A


In [14]:
import anndata

adata = anndata.read_h5ad(counts_file_path)
adata.shape

(310849, 27946)

In [15]:
list(adata.obs.index).sort() == list(metadata['barcode_sample']).sort()

True

In [16]:
pd.read_csv(degs_file_path,
            sep = '\t').head(3)

,cluster,gene,logFC,P.Val,adj.P.Val
0,Unvax_A_Macrophage,APOE,3.918956,0.0,0.0
1,Unvax_A_Macrophage,APOC1,3.841519,0.0,0.0
2,Unvax_A_Macrophage,C1QB,3.474873,0.0,0.0


In [17]:
microenv = pd.read_csv(microenvs_file_path,
                       sep = '\t')
microenv.head(3)

,major_cell_type,microenvironment
0,Unvax_A_CD8,Env1
1,Unvax_A_Macrophage,Env1
2,Unvax_A_CD4,Env1


In [18]:
microenv.groupby('microenvironment')['major_cell_type'].apply(lambda x : list(x.value_counts().index))

microenvironment
Env1    [Unvax_A_CD8, Unvax_A_Macrophage, Unvax_A_CD4,...
Name: major_cell_type, dtype: object

In [19]:
out_path

'All_sample_results/Unvaxed/method2'

In [20]:
###ADD METHOD2 FOR UNvax_A group for p value Sep20th 2023
from cellphonedb.src.core.methods import cpdb_statistical_analysis_method

deconvoluted, means, pvalues, significant_means = cpdb_statistical_analysis_method.call(
    cpdb_file_path = cpdb_file_path,                 # mandatory: CellPhoneDB database zip file.
    meta_file_path = meta_file_path,                 # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = counts_file_path,             # mandatory: normalized count matrix.
    counts_data = 'hgnc_symbol',                     # defines the gene annotation in counts matrix.
    microenvs_file_path = microenvs_file_path,       # optional (default: None): defines cells per microenvironment.
    iterations = 1000,                               # denotes the number of shufflings performed in the analysis.
    threshold = 0.1,                                 # defines the min % of cells expressing a gene for this to be employed in the analysis.
    threads = 4,                                     # number of threads to use in the analysis.
    debug_seed = 42,                                 # debug randome seed. To disable >=0.
    result_precision = 3,                            # Sets the rounding for the mean values in significan_means.
    pvalue = 0.05,                                   # P-value threshold to employ for significance.
    subsampling = False,                             # To enable subsampling the data (geometri sketching).
    subsampling_log = False,                         # (mandatory) enable subsampling log1p for non log-transformed data inputs.
    subsampling_num_pc = 100,                        # Number of componets to subsample via geometric skectching (dafault: 100).
    subsampling_num_cells = 1000,                    # Number of cells to subsample (integer) (default: 1/3 of the dataset).
    separator = '|',                                 # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    debug = False,                                   # Saves all intermediate tables employed during the analysis in pkl format.
    output_path = out_path,                          # Path to save results.
    output_suffix = None                             # Replaces the timestamp in the output files by a user defined string in the  (default: None).
    )

Reading user files...
The following user files were loaded successfully:
All_samples/all_Samples_seurat_mergered_patient_all_Samples_f.list_NFSR_Integ_1_Feb17th2023_annoated_June1st.h5ad
All_samples/Unvax_A/All_samples_Unvaxed_filtered_meta_June7th.tsv
All_samples/Unvax_A/All_samples_UA_microenv_June7th.tsv
[ ][CORE][20/09/23-12:17:08][INFO] [Cluster Statistical Analysis] Threshold:0.1 Iterations:1000 Debug-seed:42 Threads:4 Precision:3
[ ][CORE][20/09/23-12:17:09][WARNING] Debug random seed enabled. Set to 42
[ ][CORE][20/09/23-12:17:17][INFO] Running Real Analysis
[ ][CORE][20/09/23-12:17:17][INFO] Limiting cluster combinations using microenvironments
[ ][CORE][20/09/23-12:17:17][INFO] Running Statistical Analysis


100%|███████████████████████████████████████| 1000/1000 [19:56<00:00,  1.20s/it]


[ ][CORE][20/09/23-12:37:16][INFO] Building Pvalues result
[ ][CORE][20/09/23-12:37:16][INFO] Building results
Saved deconvoluted to All_sample_results/Unvaxed/method2/statistical_analysis_deconvoluted_09_20_2023_12:37:17.txt
Saved means to All_sample_results/Unvaxed/method2/statistical_analysis_means_09_20_2023_12:37:17.txt
Saved pvalues to All_sample_results/Unvaxed/method2/statistical_analysis_pvalues_09_20_2023_12:37:17.txt
Saved significant_means to All_sample_results/Unvaxed/method2/statistical_analysis_significant_means_09_20_2023_12:37:17.txt


In [67]:
ls -lt All_sample_results/Unvaxed/

total 5504
-rw-r--r-- 1 xul8 dcr_sp  530314 Jun  7 13:10 degs_analysis_significant_means_06_07_2023_13:10:21.txt
-rw-r--r-- 1 xul8 dcr_sp   84477 Jun  7 13:10 degs_analysis_relevant_interactions_result_06_07_2023_13:10:21.txt
-rw-r--r-- 1 xul8 dcr_sp 1504290 Jun  7 13:10 degs_analysis_means_result_06_07_2023_13:10:21.txt
-rw-r--r-- 1 xul8 dcr_sp  642020 Jun  7 13:10 degs_analysis_deconvoluted_result_06_07_2023_13:10:21.txt
-rw-r--r-- 1 xul8 dcr_sp  530420 Jun  7 12:19 degs_analysis_significant_means_06_07_2023_12:19:49.txt
-rw-r--r-- 1 xul8 dcr_sp   84289 Jun  7 12:19 degs_analysis_relevant_interactions_result_06_07_2023_12:19:49.txt
-rw-r--r-- 1 xul8 dcr_sp 1504290 Jun  7 12:19 degs_analysis_means_result_06_07_2023_12:19:49.txt
-rw-r--r-- 1 xul8 dcr_sp  642020 Jun  7 12:19 degs_analysis_deconvoluted_result_06_07_2023_12:19:49.txt


In [93]:
relevant_interactions=pd.read_csv("All_sample_results/Unvaxed/degs_analysis_relevant_interactions_result_06_07_2023_14:42:46.txt", delimiter = "\t")
deconvoluted=pd.read_csv("All_sample_results/Unvaxed/degs_analysis_deconvoluted_result_06_07_2023_14:42:46.txt", delimiter = "\t")
means=pd.read_csv("All_sample_results/Unvaxed/degs_analysis_means_result_06_07_2023_14:42:46.txt", delimiter = "\t")
significant_means=pd.read_csv("All_sample_results/Unvaxed/degs_analysis_significant_means_06_07_2023_14:42:46.txt", delimiter = "\t")

In [94]:
relevant_interactions.head(3)

,id_cp_interaction,interacting_pair,partner_a,partner_b,gene_a,gene_b,secreted,receptor_a,receptor_b,annotation_strategy,...,Unvax_A_unknown|Unvax_A_Macrophage,Unvax_A_unknown|Unvax_A_CD4,Unvax_A_unknown|Unvax_A_NK,Unvax_A_unknown|Unvax_A_Epithelial,Unvax_A_unknown|Unvax_A_Plasma,Unvax_A_unknown|Unvax_A_mDC,Unvax_A_unknown|Unvax_A_Mast,Unvax_A_unknown|Unvax_A_B_cell,Unvax_A_unknown|Unvax_A_pDC,Unvax_A_unknown|Unvax_A_unknown
0,CPI-SS0F66EA6BE,APP_FPR2,simple:P05067,simple:P25090,APP,FPR2,False,False,True,curated,...,0,0,0,0,0,0,0,0,0,0
1,CPI-SS05B2931D3,CCL23_FPR2,simple:P55773,simple:P25090,CCL23,FPR2,True,False,True,curated,...,0,0,0,0,0,0,0,0,0,0
2,CPI-CS0FC36F030,LipoxinA4_byALOX5_FPR2,complex:LipoxinA4_byALOX5,simple:P25090,NaN,FPR2,True,False,True,curated,...,0,0,0,0,0,0,0,0,0,0


In [95]:
significant_means.head(4)

,id_cp_interaction,interacting_pair,partner_a,partner_b,gene_a,gene_b,secreted,receptor_a,receptor_b,annotation_strategy,...,Unvax_A_unknown|Unvax_A_Macrophage,Unvax_A_unknown|Unvax_A_CD4,Unvax_A_unknown|Unvax_A_NK,Unvax_A_unknown|Unvax_A_Epithelial,Unvax_A_unknown|Unvax_A_Plasma,Unvax_A_unknown|Unvax_A_mDC,Unvax_A_unknown|Unvax_A_Mast,Unvax_A_unknown|Unvax_A_B_cell,Unvax_A_unknown|Unvax_A_pDC,Unvax_A_unknown|Unvax_A_unknown
0,CPI-SS000568A0E,FCER2_CR2,simple:P06734,simple:P20023,FCER2,CR2,True,True,True,curated,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CPI-SS00AEC3818,CCL19_ACKR4,simple:Q99731,simple:Q9NPB9,CCL19,ACKR4,True,False,True,curated,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CPI-SS052B38F1C,CCL22_CCR4,simple:O00626,simple:P51679,CCL22,CCR4,True,False,True,curated,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,CPI-SS07A1648FA,NRXN3_CLSTN2,simple:Q9HDB5,simple:Q9H4D0,NRXN3,CLSTN2,False,False,False,curated,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [96]:
means.head(3)

,id_cp_interaction,interacting_pair,partner_a,partner_b,gene_a,gene_b,secreted,receptor_a,receptor_b,annotation_strategy,...,Unvax_A_unknown|Unvax_A_Macrophage,Unvax_A_unknown|Unvax_A_CD4,Unvax_A_unknown|Unvax_A_NK,Unvax_A_unknown|Unvax_A_Epithelial,Unvax_A_unknown|Unvax_A_Plasma,Unvax_A_unknown|Unvax_A_mDC,Unvax_A_unknown|Unvax_A_Mast,Unvax_A_unknown|Unvax_A_B_cell,Unvax_A_unknown|Unvax_A_pDC,Unvax_A_unknown|Unvax_A_unknown
0,CPI-CS0A5B6BD7A,12oxoLeukotrieneB4_byPTGR1_LTB4R,complex:12oxoLeukotrieneB4_byPTGR1,simple:Q15722,NaN,LTB4R,True,False,True,curated,...,0.052,0.043,0.041,0.045,0.031,0.039,0.047,0.035,0.043,0.071
1,CPI-CS047D9C0D7,LeukotrieneB4_byLTA4H_LTB4R,complex:LeukotrieneB4_byLTA4H,simple:Q15722,NaN,LTB4R,True,False,True,curated,...,1.112,1.102,1.101,1.104,1.091,1.099,1.106,1.095,1.103,1.131
2,CPI-CS04A56D5BE,12oxoLeukotrieneB4_byPTGR1_LTB4R2,complex:12oxoLeukotrieneB4_byPTGR1,simple:Q9NPC1,NaN,LTB4R2,True,False,True,curated,...,0.031,0.049,0.038,0.055,0.027,0.034,0.046,0.030,0.033,0.000


In [97]:
deconvoluted.head(3)

,gene_name,uniprot,is_complex,protein_name,complex_name,id_cp_interaction,Unvax_A_B_cell,Unvax_A_CD4,Unvax_A_CD8,Unvax_A_Epithelial,Unvax_A_Macrophage,Unvax_A_Mast,Unvax_A_NK,Unvax_A_Plasma,Unvax_A_mDC,Unvax_A_pDC,Unvax_A_unknown
0,UBASH3B,Q8TF42,True,UBS3B_HUMAN,Dehydroepiandrosterone_bySTS,CPI-CS09B8977D7,0.02,0.172,0.134,0.107,0.421,0.416,0.263,0.06,0.206,0.12,0.098
1,UBASH3B,Q8TF42,True,UBS3B_HUMAN,Dehydroepiandrosterone_bySTS,CPI-CS05760BB78,0.02,0.172,0.134,0.107,0.421,0.416,0.263,0.06,0.206,0.12,0.098
2,UBASH3B,Q8TF42,True,UBS3B_HUMAN,Dehydroepiandrosterone_bySTS,CPI-CS0259A0EB4,0.02,0.172,0.134,0.107,0.421,0.416,0.263,0.06,0.206,0.12,0.098


In [ ]:
######GET Vaxed A cellphoneDBJUNE7TH

In [21]:
mkdir All_sample_results/VaxedA/method2

In [22]:
####Cd8 SUCCESSFULLY GET THE CELLPHONEDB
cpdb_file_path = 'v4.1.0/cellphonedb.zip'
meta_file_path = 'All_samples/Vaxed_A/All_samples_Vaxed_filtered_meta_June7th.tsv'
counts_file_path = 'All_samples/all_Samples_seurat_counts.h5ad'
microenvs_file_path = 'All_samples/Vaxed_A/All_samples_VaxedA_microenv_June7th.tsv'
degs_file_path = 'All_samples/Vaxed_A/All_Samples_Vaxed_aLL_markers_06_07_2023.tsv'
out_path = 'All_sample_results/VaxedA/method2'

In [7]:
metadata = pd.read_csv(meta_file_path, sep = '\t')
metadata.head(3)

#(metadata)['major_cell_type']
#metadata["major_cell_type"][1:3]

,barcode_sample,major_cell_type,state
0,11_AC1_1_7_22_AAACCTGAGATCGGGT-1,Vaxed_A_Macrophage,Vaxed_A
1,11_AC1_1_7_22_AAACCTGAGCGAAGGG-1,Vaxed_A_Macrophage,Vaxed_A
2,11_AC1_1_7_22_AAACCTGAGTGTCTCA-1,Vaxed_A_CD4,Vaxed_A


In [8]:
import anndata

adata = anndata.read_h5ad(counts_file_path)
adata.shape

(310849, 27946)

In [23]:
list(adata.obs.index).sort() == list(metadata['barcode_sample']).sort()

True

In [10]:
pd.read_csv(degs_file_path,
            sep = '\t').head(3)

,cluster,gene,logFC,P.Val,adj.P.Val
0,Vaxed_A_Macrophage,APOE,4.678050,0.0,0.0
1,Vaxed_A_Macrophage,APOC1,4.570257,0.0,0.0
2,Vaxed_A_Macrophage,C1QB,4.158860,0.0,0.0


In [11]:
microenv = pd.read_csv(microenvs_file_path,
                       sep = '\t')
microenv.head(3)

,major_cell_type,microenvironment
0,Vaxed_A_Macrophage,Env1
1,Vaxed_A_CD4,Env1
2,Vaxed_A_CD8,Env1


In [24]:
###ADD METHOD2 FOR UNvax_A group for p value Sep20th 2023
from cellphonedb.src.core.methods import cpdb_statistical_analysis_method

deconvoluted, means, pvalues, significant_means = cpdb_statistical_analysis_method.call(
    cpdb_file_path = cpdb_file_path,                 # mandatory: CellPhoneDB database zip file.
    meta_file_path = meta_file_path,                 # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = counts_file_path,             # mandatory: normalized count matrix.
    counts_data = 'hgnc_symbol',                     # defines the gene annotation in counts matrix.
    microenvs_file_path = microenvs_file_path,       # optional (default: None): defines cells per microenvironment.
    iterations = 1000,                               # denotes the number of shufflings performed in the analysis.
    threshold = 0.1,                                 # defines the min % of cells expressing a gene for this to be employed in the analysis.
    threads = 4,                                     # number of threads to use in the analysis.
    debug_seed = 42,                                 # debug randome seed. To disable >=0.
    result_precision = 3,                            # Sets the rounding for the mean values in significan_means.
    pvalue = 0.05,                                   # P-value threshold to employ for significance.
    subsampling = False,                             # To enable subsampling the data (geometri sketching).
    subsampling_log = False,                         # (mandatory) enable subsampling log1p for non log-transformed data inputs.
    subsampling_num_pc = 100,                        # Number of componets to subsample via geometric skectching (dafault: 100).
    subsampling_num_cells = 1000,                    # Number of cells to subsample (integer) (default: 1/3 of the dataset).
    separator = '|',                                 # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    debug = False,                                   # Saves all intermediate tables employed during the analysis in pkl format.
    output_path = out_path,                          # Path to save results.
    output_suffix = None                             # Replaces the timestamp in the output files by a user defined string in the  (default: None).
    )

Reading user files...
The following user files were loaded successfully:
All_samples/all_Samples_seurat_mergered_patient_all_Samples_f.list_NFSR_Integ_1_Feb17th2023_annoated_June1st.h5ad
All_samples/Vaxed_A/All_samples_Vaxed_filtered_meta_June7th.tsv
All_samples/Vaxed_A/All_samples_VaxedA_microenv_June7th.tsv
[ ][CORE][20/09/23-12:39:37][INFO] [Cluster Statistical Analysis] Threshold:0.1 Iterations:1000 Debug-seed:42 Threads:4 Precision:3
[ ][CORE][20/09/23-12:39:37][WARNING] Debug random seed enabled. Set to 42
[ ][CORE][20/09/23-12:39:46][INFO] Running Real Analysis
[ ][CORE][20/09/23-12:39:46][INFO] Limiting cluster combinations using microenvironments
[ ][CORE][20/09/23-12:39:46][INFO] Running Statistical Analysis


100%|███████████████████████████████████████| 1000/1000 [20:20<00:00,  1.22s/it]


[ ][CORE][20/09/23-13:00:09][INFO] Building Pvalues result
[ ][CORE][20/09/23-13:00:09][INFO] Building results
Saved deconvoluted to All_sample_results/VaxedA/method2/statistical_analysis_deconvoluted_09_20_2023_13:00:10.txt
Saved means to All_sample_results/VaxedA/method2/statistical_analysis_means_09_20_2023_13:00:10.txt
Saved pvalues to All_sample_results/VaxedA/method2/statistical_analysis_pvalues_09_20_2023_13:00:10.txt
Saved significant_means to All_sample_results/VaxedA/method2/statistical_analysis_significant_means_09_20_2023_13:00:10.txt
